In [1]:
import glob
import os
import pandas as pd

# 1. Đọc dữ liệu từ Kaggle Dataset read-unified-pool-v1
pool_files = glob.glob("/kaggle/input/**/read_unified_pool.parquet", recursive=True)
assert len(pool_files) > 0, "Không tìm thấy read_unified_pool.parquet! Hãy kiểm tra mục Add Input."

df_pool = pd.read_parquet(pool_files[0])
print(f"Tổng dữ liệu gốc: {len(df_pool):,} mẫu")

# Tạo cột id duy nhất để kiểm soát rò rỉ dữ liệu tuyệt đối
df_pool = df_pool.reset_index(drop=True)
df_pool["sample_id"] = df_pool.index

# 2. Tách tập Test độc lập 9,600 mẫu (cân bằng 50% Human / 50% AI theo 11 miền)
test_quota = {
    "CMV": 641,
    "ELI5": 779,
    "HELLASWAG": 778,
    "ROC": 805,
    "SCI": 509,
    "SQUAD": 551,
    "TLDR": 630,
    "WP": 798,
    "XSUM": 771,
    "YELP": 560,
    "REWRITE": 2778,
}

test_parts = []
for dom, total_n in test_quota.items():
    sub = df_pool[df_pool["domain"] == dom]
    n_human = total_n // 2
    n_ai = total_n - n_human

    s_human = sub[sub["label"] == "Human"].sample(n=n_human, random_state=42)
    s_ai = sub[sub["label"] == "AI"].sample(n=n_ai, random_state=42)
    test_parts.extend([s_human, s_ai])

df_test = pd.concat(test_parts, ignore_index=True).sample(frac=1.0, random_state=42).reset_index(drop=True)

# Loại bỏ toàn bộ mẫu đã vào tập Test khỏi pool huấn luyện
test_ids = set(df_test["sample_id"])
df_remaining = df_pool[~df_pool["sample_id"].isin(test_ids)].reset_index(drop=True)

# 3. Trích tập Train chuẩn (44,000 mẫu = 4,000 mẫu/miền, cân bằng 50/50)
# Lưu ý: Có miền như TLDR Human còn ~2,511 mẫu sau khi trừ Test, nên lấy tối đa min(2000, số mẫu hiện có) để đảm bảo cân bằng 50/50 từng miền
train_parts = []
for dom in sorted(df_remaining["domain"].unique()):
    sub = df_remaining[df_remaining["domain"] == dom]
    n_human_avail = (sub["label"] == "Human").sum()
    n_ai_avail = (sub["label"] == "AI").sum()
    n_per_class = min(2000, n_human_avail, n_ai_avail)

    s_human = sub[sub["label"] == "Human"].sample(n=n_per_class, random_state=42)
    s_ai = sub[sub["label"] == "AI"].sample(n=n_per_class, random_state=42)
    train_parts.extend([s_human, s_ai])

df_train_all = pd.concat(train_parts, ignore_index=True).sample(frac=1.0, random_state=42).reset_index(drop=True)

# 4. Trích tập con cho Teacher sinh chuỗi suy luận (1,000 mẫu/miền = 11,000 mẫu cân bằng 50/50)
teacher_parts = []
for dom in sorted(df_train_all["domain"].unique()):
    sub = df_train_all[df_train_all["domain"] == dom]
    s_human = sub[sub["label"] == "Human"].sample(n=500, random_state=42)
    s_ai = sub[sub["label"] == "AI"].sample(n=500, random_state=42)
    teacher_parts.extend([s_human, s_ai])

df_teacher_sub = pd.concat(teacher_parts, ignore_index=True).sample(frac=1.0, random_state=42).reset_index(drop=True)

# Phần còn lại của df_train_all (dùng bổ sung cho GRPO/DPO sau khi lọc SFT)
teacher_ids = set(df_teacher_sub["sample_id"])
df_rl_extra = df_train_all[~df_train_all["sample_id"].isin(teacher_ids)].reset_index(drop=True)

# 5. Kiểm tra tính toàn vẹn (Zero Leakage) & Lưu file
assert len(set(df_test["sample_id"]) & set(df_train_all["sample_id"])) == 0, "Lỗi rò rỉ dữ liệu Test-Train!"
assert len(set(df_test["text"]) & set(df_train_all["text"])) == 0, "Lỗi trùng văn bản Test-Train!"

out_dir = "/kaggle/working/read_splits"
os.makedirs(out_dir, exist_ok=True)

df_test.to_parquet(f"{out_dir}/test_9600.parquet", index=False)
df_train_all.to_parquet(f"{out_dir}/train_baseline_all.parquet", index=False)
df_teacher_sub.to_parquet(f"{out_dir}/teacher_sub_11k.parquet", index=False)
df_rl_extra.to_parquet(f"{out_dir}/rl_extra_pool.parquet", index=False)

print("\n=== TỔNG KẾT CHIA DỮ LIỆU GIAI ĐOẠN 1 ===")
print(f"1. Tập Test độc lập : {len(df_test):,} mẫu | Human: {(df_test['label']=='Human').sum():,} | AI: {(df_test['label']=='AI').sum():,}")
print(f"2. Tập Train tổng   : {len(df_train_all):,} mẫu | Human: {(df_train_all['label']=='Human').sum():,} | AI: {(df_train_all['label']=='AI').sum():,}")
print(f"3. Tập Teacher (SFT): {len(df_teacher_sub):,} mẫu | Human: {(df_teacher_sub['label']=='Human').sum():,} | AI: {(df_teacher_sub['label']=='AI').sum():,}")
print(f"4. Tập RL bổ sung   : {len(df_rl_extra):,} mẫu | Human: {(df_rl_extra['label']=='Human').sum():,} | AI: {(df_rl_extra['label']=='AI').sum():,}")
print("\nPhân bố tập Test theo miền:")
print(pd.crosstab(df_test["domain"], df_test["label"], margins=True))

Tổng dữ liệu gốc: 338,666 mẫu

=== TỔNG KẾT CHIA DỮ LIỆU GIAI ĐOẠN 1 ===
1. Tập Test độc lập : 9,600 mẫu | Human: 4,797 | AI: 4,803
2. Tập Train tổng   : 44,000 mẫu | Human: 22,000 | AI: 22,000
3. Tập Teacher (SFT): 11,000 mẫu | Human: 5,500 | AI: 5,500
4. Tập RL bổ sung   : 33,000 mẫu | Human: 16,500 | AI: 16,500

Phân bố tập Test theo miền:
label        AI  Human   All
domain                      
CMV         321    320   641
ELI5        390    389   779
HELLASWAG   389    389   778
REWRITE    1389   1389  2778
ROC         403    402   805
SCI         255    254   509
SQUAD       276    275   551
TLDR        315    315   630
WP          399    399   798
XSUM        386    385   771
YELP        280    280   560
All        4803   4797  9600
